<a href="https://colab.research.google.com/github/TTS-77/Cyclone-AI/blob/main/Notebook/Formatting_and_Splitting_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Formatting and Splitting Dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "IBTrACS_ERA5_NI_2000_2025.csv"
)

df["ISO_TIME"] = pd.to_datetime(df["ISO_TIME"])

df = df.sort_values(
    ["SID", "ISO_TIME"]
).reset_index(drop=True)

print("Rows:", len(df))
print("Cyclones:", df["SID"].nunique())

Rows: 8817
Cyclones: 264


In [ ]:
df["temp_dewpoint_diff"] = (
    df["t2m_C"] - df["d2m_C"]
)

In [ ]:
df["prev_lat"] = df.groupby("SID")["LAT"].shift(1)
df["prev_lon"] = df.groupby("SID")["LON"].shift(1)
df["prev_time"] = df.groupby("SID")["ISO_TIME"].shift(1)

df["time_diff_hours"] = (
    df["ISO_TIME"] - df["prev_time"]
).dt.total_seconds() / 3600

# Approximate distance in km
lat1 = np.radians(df["prev_lat"])
lat2 = np.radians(df["LAT"])

dlat = np.radians(
    df["LAT"] - df["prev_lat"]
)

dlon = np.radians(
    df["LON"] - df["prev_lon"]
)

a = (
    np.sin(dlat / 2)**2 +
    np.cos(lat1) *
    np.cos(lat2) *
    np.sin(dlon / 2)**2
)

earth_radius = 6371

df["movement_distance_km"] = (
    2 *
    earth_radius *
    np.arcsin(np.sqrt(a))
)

df["movement_speed_kmh"] = (
    df["movement_distance_km"] /
    df["time_diff_hours"]
)

In [ ]:
df = df.drop(
    columns=[
        "prev_lat",
        "prev_lon",
        "prev_time"
    ]
)

In [ ]:
df["target_wind"] = df["USA_WIND"]

In [ ]:
df["target_pressure"] = df["USA_PRES"]

In [ ]:
# ===== CORRECT CHRONOLOGICAL SPLIT =====

train_df = df[
    df["SEASON"].between(2000, 2018)
].copy()

val_df = df[
    df["SEASON"].between(2019, 2021)
].copy()

test_df = df[
    df["SEASON"].between(2022, 2025)
].copy()

In [ ]:
train_df.to_csv(
    "cyclone_train_2000_2018.csv",
    index=False
)

val_df.to_csv(
    "cyclone_validation_2019_2021.csv",
    index=False
)

test_df.to_csv(
    "cyclone_test_2022_2025.csv",
    index=False
)

print("Corrected datasets saved successfully.")

Corrected datasets saved successfully.


In [ ]:
print("========== FINAL SPLIT CHECK ==========\n")

print("TRAIN")
print("Rows:", len(train_df))
print("Cyclones:", train_df["SID"].nunique())
print("Seasons:", train_df["SEASON"].min(), "to", train_df["SEASON"].max())

print("\nVALIDATION")
print("Rows:", len(val_df))
print("Cyclones:", val_df["SID"].nunique())
print("Seasons:", val_df["SEASON"].min(), "to", val_df["SEASON"].max())

print("\nTEST")
print("Rows:", len(test_df))
print("Cyclones:", test_df["SID"].nunique())
print("Seasons:", test_df["SEASON"].min(), "to", test_df["SEASON"].max())

print("\n========== OVERLAPPING SEASONS ==========")

print("Train seasons:", sorted(train_df["SEASON"].unique()))
print("Validation seasons:", sorted(val_df["SEASON"].unique()))
print("Test seasons:", sorted(test_df["SEASON"].unique()))

========== FINAL SPLIT CHECK ==========

TRAIN
Rows: 6009
Cyclones: 178
Seasons: 2000 to 2018

VALIDATION
Rows: 1198
Cyclones: 31
Seasons: 2019 to 2021

TEST
Rows: 1610
Cyclones: 55
Seasons: 2022 to 2025

========== OVERLAPPING SEASONS ==========
Train seasons: [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
Validation seasons: [np.int64(2019), np.int64(2020), np.int64(2021)]
Test seasons: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
